In [1]:
#r "nuget: Azure.AI.Projects, 1.2.0-beta.5"
#r "nuget: Azure.Identity, 1.18.0-beta.2"
#r "nuget: Microsoft.Agents.AI, 1.0.0-rc1"
#r "nuget: Microsoft.Agents.AI.AzureAI, 1.0.0-rc1"
#r "nuget: Microsoft.Agents.AI.Workflows, 1.0.0-rc1"

using System.IO;
using Azure.AI.Projects;
using Azure.Identity;
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using Microsoft.Agents.AI.Workflows;

var tenantId = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_TENANT_ID");
var endpointUrl = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_ENDPOINT");
var deploymentName = "gpt-4o";

var credOptions = new DefaultAzureCredentialOptions
{
    TenantId = tenantId    
};

// Connect to your project using the endpoint from your project page
AIProjectClient projectClient = new(
    new Uri(endpointUrl),
    new DefaultAzureCredential(credOptions));

// Define Concierge Agent (Quality Reviewer)
const string REVIEWER_NAME = "Concierge";
const string REVIEWER_INSTRUCTIONS = @"""
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """;

// Define Front Desk Agent (Travel Specialist)
const string FRONTDESK_NAME = "FrontDesk";
const string FRONTDESK_INSTRUCTIONS = @"""
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """;

// Configure agent options
ChatClientAgentOptions frontdeskAgentOptions = new()
{
    Name = FRONTDESK_NAME,
    Description = FRONTDESK_INSTRUCTIONS
};

ChatClientAgentOptions reviewerAgentOptions = new()
{
    Name = REVIEWER_NAME,
    Description = REVIEWER_INSTRUCTIONS
};

// Create agent instances
AIAgent reviewerAgent = await projectClient.CreateAIAgentAsync(reviewerAgentOptions.Name, deploymentName, reviewerAgentOptions.Description);
AIAgent frontdeskAgent = await projectClient.CreateAIAgentAsync(frontdeskAgentOptions.Name, deploymentName, frontdeskAgentOptions.Description);

// Build the multi-agent workflow
var workflow = new WorkflowBuilder(frontdeskAgent)
    .AddEdge(frontdeskAgent, reviewerAgent)
    .Build();

// Start the streaming workflow execution
StreamingRun run = await InProcessExecution.RunStreamingAsync(workflow, new ChatMessage(ChatRole.User, "I would like to go to Paris."));

// Send message to start workflow
await run.TrySendMessageAsync(new TurnToken(emitEvents: true));

// Process streaming workflow events
string strResult = "";
await foreach (WorkflowEvent evt in run.WatchStreamAsync().ConfigureAwait(false))
{
    if (evt is AgentResponseUpdateEvent executorComplete)
    {
        strResult += executorComplete.Data;
        Console.WriteLine($"{executorComplete.ExecutorId}: {executorComplete.Data}");
    }
}

// Display final aggregated result
Console.WriteLine("\n=== Final Result ===");
Console.WriteLine(strResult);

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Azure.AI.Projects, 1.2.0-beta.5 Azure.Identity, 1.18.0-beta.2 Microsoft.Agents.AI, 1.0.0-rc1 Microsoft.Agents.AI.AzureAI, 1.0.0-rc1 Microsoft.Agents.AI.Workflows, 1.0.0-rc1

FrontDesk_FrontDesk_2: 
FrontDesk_FrontDesk_2: 
FrontDesk_FrontDesk_2: 
FrontDesk_FrontDesk_2: 
FrontDesk_FrontDesk_2: Visit
FrontDesk_FrontDesk_2:  the
FrontDesk_FrontDesk_2:  Louvre
FrontDesk_FrontDesk_2:  Museum
FrontDesk_FrontDesk_2: .
FrontDesk_FrontDesk_2: 
FrontDesk_FrontDesk_2: 
FrontDesk_FrontDesk_2: 
FrontDesk_FrontDesk_2: 
Concierge_Concierge_2: 
Concierge_Concierge_2: 
Concierge_Concierge_2: 
Concierge_Concierge_2: 
Concierge_Concierge_2: While
Concierge_Concierge_2:  the
Concierge_Concierge_2:  Louvre
Concierge_Concierge_2:  is
Concierge_Concierge_2:  iconic
Concierge_Concierge_2:  and
Concierge_Concierge_2:  a
Concierge_Concierge_2:  must
Concierge_Concierge_2: -see
Concierge_Concierge_2:  for
Concierge_Concierge_2:  many
Concierge_Concierge_2: ,
Concierge_Concierge_2:  it's
Concierge_Concierge_2:  quite
Concierge_Concierge_2:  tourist
Concierge_Concierge_2: y
Concierge_Concierge_2: .
Concierge_Concierge_2:  For
Concierge_Concierge_2:  a
Concierge_Concierge_2:  more
Conci


warning CS1702: Assuming assembly reference 'System.ClientModel, Version=1.8.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.Core' matches identity 'System.ClientModel, Version=1.8.1.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy

